Starting Day 1 on 5/25/2026 

In [1]:
import matplotlib.pyplot as plt

In [2]:
words = open('names.txt', 'r').read().splitlines()


In [8]:
words[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [4]:
print(min(len(w) for w in words))
print(max(len(w) for w in words))

2
15


First we build the Bigram language model. We will be working with a list of names, and we want to learn how to generate new names that look similar to the ones in the list.

In [3]:
b = {}
for w in words[:]:
    chs = ['<S'] + list(w) + ['<E']
    for ch1, ch2 in zip(chs, chs[1:]):
        bigram = (ch1, ch2)
        b[bigram] = b.get(bigram, 0) + 1
# To figure out how common the characters are next to each other, we are going to count!

In [22]:
# sorted(b.items(), key= lambda kv: -kv[1]) 
# It's going to be convenient for us to use a 2D array instead of a dict

In [4]:
import torch
a = torch.zeros((3, 5), dtype= torch.int32) # since we will only use counts, int is better
a

tensor([[0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0]], dtype=torch.int32)

In [7]:
# we require a bigger array, that contains all alphabets + two new characters (<S and <E)
N = torch.zeros((28, 28), dtype = torch.int32)

We need a way to convert the characters to numbers

In [5]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i for i, s in enumerate(chars)} # enumerate gives index and the element in that index
stoi['<S'] = 26
stoi['<E'] = 27

In [8]:
for w in words[:]:
    chs = ['<S'] + list(w) + ['<E']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        N[ix1, ix2] += 1

In [53]:
N[3, 3].item() # gives the direct integer instead of a tensor

149

In [ ]:
itos = {i:s for s, i in stoi.items()}
plt.figure(figsize= (16, 16))
plt.imshow(N, cmap= 'Blues')
for i in range(28):
    for j in range(28):
        chstr = itos[i] + itos[j]
        plt.text(j, i, chstr, ha='center', va='bottom', color= 'blue')
        plt.text(j, i, N[i,j].item(), ha='center', va='top', color= 'red') 
plt.axis('off')

Observe carefully to see that we have a row of 0s where E being the starting char, which never happens and same applied to S being the ending char. Next, we have to tidy up this matrix.

Done Day 1 on 5/25/2026 at 19:53 -> video time (22:10)

Starting Day 2 on 5/26/2026 at 17:52

(28, 28) matrix will be changed to (27, 27) to remove the extra unused character.

In [5]:
N = torch.zeros((27, 27), dtype = torch.int32)

In [6]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)} # enumerate gives index and the element in that index
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}
# itos

In [7]:
for w in words[:]:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        N[ix1, ix2] += 1

In [ ]:
itos = {i:s for s, i in stoi.items()}
plt.figure(figsize= (16, 16))
plt.imshow(N, cmap= 'Blues')
for i in range(27):
    for j in range(27):
        chstr = itos[i] + itos[j]
        plt.text(j, i, chstr, ha='center', va='bottom', color= 'blue')
        plt.text(j, i, N[i,j].item(), ha='center', va='top', color= 'red') 
plt.axis('off')

In [8]:
N[0]

tensor([   0, 4410, 1306, 1542, 1690, 1531,  417,  669,  874,  591, 2422, 2963,
        1572, 2538, 1146,  394,  515,   92, 1639, 2055, 1308,   78,  376,  307,
         134,  535,  929], dtype=torch.int32)

We want to convert these counts to probabilities to sample them

In [9]:
p = N[0].float() # convert to float for division
p = p / p.sum() # normalize to get probabilities
p

tensor([0.0000, 0.1377, 0.0408, 0.0481, 0.0528, 0.0478, 0.0130, 0.0209, 0.0273,
        0.0184, 0.0756, 0.0925, 0.0491, 0.0792, 0.0358, 0.0123, 0.0161, 0.0029,
        0.0512, 0.0642, 0.0408, 0.0024, 0.0117, 0.0096, 0.0042, 0.0167, 0.0290])

In [10]:
g = torch.Generator().manual_seed(2147483647) # set the seed for reproducibility
ix = torch.multinomial(p, num_samples = 1, replacement = True, generator=g).item()
itos[ix]

'c'

In [11]:
p.sum() # the sum of probabilities should be 1 

tensor(1.)

Finishd Day 2 on 5/26/2026 at 18:06 -> video time (27:50)

Starting Day 3 on 5/27/2026 at 18:00

In [12]:
g = torch.Generator().manual_seed(2147483647) # set the seed for reproducibility
p = torch.rand(3, generator=g)
p = p/p.sum()
p

tensor([0.6064, 0.3033, 0.0903])

In [14]:
torch.multinomial(p, num_samples = 3, replacement = True) # gives us 3 samples from the distribution p, with replacement

tensor([1, 0, 2])

Something to notice here is that, we are sampling from the distribution p, and by carefully looking at it's output, you have a 60% chance of getting a 0, 30 % for 1, and 10% for 2. And that's what happens in multinomial sampling!

In [15]:
g = torch.Generator().manual_seed(2147483647) # set the seed for reproducibility
for i in range(20):
    
    out = []
    ix = 0
    while True:
        p = N[ix].float()
        p = p/ p.sum()
        ix = torch.multinomial(p, num_samples = 1, replacement = True, generator=g).item()
        out.append(itos[ix]) 
        if ix == 0:
            break 
    print("".join(out))

cexze.
momasurailezitynn.
konimittain.
llayn.
ka.
da.
staiyaubrtthrigotai.
moliellavo.
ke.
teda.
ka.
emimmsade.
enkaviyny.
ftlspihinivenvorhlasu.
dsor.
br.
jol.
pen.
aisan.
ja.


Finished Day 3 on 5/27/2026 at 18:34 -> video time (33:24)

Starting Day 4 on 5/28/2026 at 12:00

In [16]:
P = N.float()
P.sum() # If you sum it this way, it's adding up all the count across the N matrix, but we want to add values across the row
# because we know that their sum of probabilities will equal 1. 

tensor(228146.)

In [17]:
P.sum(0, keepdim= True).shape # this sums up values across the columns, so you get a row vector

torch.Size([1, 27])

In [18]:
P.sum(1, keepdim= True).shape # this is what we want
# but why are we saying keepdim as true? 
# It's because if it's false, it would squeeze the array to one dim. For example, 27 X 1, would become just 27.

torch.Size([27, 1])

In [19]:
P.sum(1).shape
# This would simply be 27 because keepdim is false.
# And when we do (27, 27) with (27) - internally it would go as
# 27, 27
#   , 27 -> (1, 27)
# so instead of column vector, it will be a row vector(1 X 27) and because of that, it would normalize across columns instead of rows like we intend to

torch.Size([27])

A fun fact - The counts sum across the rows and columns is identical

In [20]:
P = (N+1).float()
P_sum = P.sum(1, keepdim= True)
# Is it possible to divide these terms? (27, 27) matrix with (27, 1). Yes it is because of broadcasting!
P /= P_sum 
P.shape

torch.Size([27, 27])

In [21]:
P[0].sum()

tensor(1.)

In [22]:
g = torch.Generator().manual_seed(2147483647) # set the seed for reproducibility
for i in range(5):
    
    out = []
    ix = 0
    while True:
        
        p = P[ix]
        #p = N[ix].float()
        #p = p/ p.sum()
        ix = torch.multinomial(p, num_samples = 1, replacement = True, generator=g).item()
        out.append(itos[ix]) 
        if ix == 0:
            break 
    print("".join(out))

cexze.
momasurailezitynn.
konimittain.
llayn.
ka.


Finished Day 4 on 5/28/2026 at 12:50 -> video time (50:22)

Starting Day 5 on 5/29/2026 at 18:37

In [ ]:
for w in words[:3]:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        prob = P[ix1, ix2] 
        print(f'{ch1}{ch2}: {prob:.4f}')

MLE says that the likelihood of the entire data is given by product of individual probabilites(which defines model quality)
So we can calculate the likelihood of the data by multiplying the probabilities of each bigram in the data.
But this is a very small number, so we can take the log of the likelihood to get a more manageable number. 
The log of the likelihood is given by the sum of the log of the probabilities of each bigram in the data.

In [24]:
log_likelihood = 0.0
n= 0
for w in words[:3]:
# for w in ["yaswanthqp"]:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        prob = P[ix1, ix2] 
        logprob = torch.log(prob)
        log_likelihood += logprob
        n += 1
        print(f'{ch1}{ch2}: {prob:.4f}')
        
print(f'log likelihood: {log_likelihood:.4f}')
# take negative log -> -log is convex, and we can minimize it but log is concave and we will maximize it. Loss is dealt in minimizing. 
nll = -log_likelihood
print(f'{nll=}')
print(f'{nll/n}') # average negative log likelihood per bigram, this is the loss that we want to minimize.

.e: 0.0478
em: 0.0377
mm: 0.0253
ma: 0.3885
a.: 0.1958
.o: 0.0123
ol: 0.0779
li: 0.1774
iv: 0.0152
vi: 0.3508
ia: 0.1380
a.: 0.1958
.a: 0.1376
av: 0.0246
va: 0.2473
a.: 0.1958
log likelihood: -38.8086
nll=tensor(38.8086)
2.4255354404449463


Normally, this would give -inf (the one with my name), but since we have added 1 to the counts, we can avoid that. This is called Laplace smoothing, and it is a common technique to avoid zero probabilities in language models.

Finished Day 5 on 5/29/2026 at 19:02 -> video time (58:21)

Starting Day 6 on 5/30/2026 at 16:25

Goal: Maxmize the likelihood(product of the probabilites) of the data w.r.t model parameters (statistical modelling)
Which is equivalent to maximizing the log likelihood (because log is monotonic) 
Equivalent to minimizing the negative log likelihood (because we want to minimize the loss)
Equivalent to minimizing average negative log likelihood

Now we are moving to building a Neural net.

In [25]:
# construct a training set
xs, ys = [], []

for w in words[:1]:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2] 
        print(ch1, ch2)
        xs.append(ix1)
        ys.append(ix2)
        
xs = torch.tensor(xs)
ys = torch.tensor(ys)

. e
e m
m m
m a
a .


In [26]:
xs, ys
# explanation: For 0 input, you'd want 5 (means 'e'), For 5 input, you want 13 ('m') and so on.
# These are indices and you will give them to one hot encoder, 
# to get 1s wherever the index is, and you convert that to floats because NN requires floats.

(tensor([ 0,  5, 13, 13,  1]), tensor([ 5, 13, 13,  1,  0]))

In [ ]:
import torch.nn.functional as F
xenc = F.one_hot(xs, num_classes = 27).float() # inputs to the NN. 

In [28]:
xenc.dtype, xenc.shape

(torch.float32, torch.Size([5, 27]))

Finished Day 6 on 5/30/2026 at 17:31 -> video time (1:13:59)

Starting Day 7 on 5/31/2026 at 17:29

In [31]:
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator= g) # randomly initialize 27 neurons' weights, each neuron receives 27 inputs.
xenc @ W # (5, 27) @ (27, 1) -> (5, 1)

tensor([[ 1.5674e+00, -2.3729e-01, -2.7385e-02, -1.1008e+00,  2.8588e-01,
         -2.9643e-02, -1.5471e+00,  6.0489e-01,  7.9136e-02,  9.0462e-01,
         -4.7125e-01,  7.8682e-01, -3.2843e-01, -4.3297e-01,  1.3729e+00,
          2.9334e+00,  1.5618e+00, -1.6261e+00,  6.7716e-01, -8.4039e-01,
          9.8488e-01, -1.4837e-01, -1.4795e+00,  4.4830e-01, -7.0730e-02,
          2.4968e+00,  2.4448e+00],
        [ 4.7236e-01,  1.4830e+00,  3.1748e-01,  1.0588e+00,  2.3982e+00,
          4.6827e-01, -6.5650e-01,  6.1662e-01, -6.2197e-01,  5.1007e-01,
          1.3563e+00,  2.3445e-01, -4.5585e-01, -1.3132e-03, -5.1161e-01,
          5.5570e-01,  4.7458e-01, -1.3867e+00,  1.6229e+00,  1.7197e-01,
          9.8846e-01,  5.0657e-01,  1.0198e+00, -1.9062e+00, -4.2753e-01,
         -2.1259e+00,  9.6041e-01],
        [ 1.9359e-01,  1.0532e+00,  6.3393e-01,  2.5786e-01,  9.6408e-01,
         -2.4855e-01,  2.4756e-02, -3.0404e-02,  1.5622e+00, -4.4852e-01,
         -1.2345e+00,  1.1220e+00, -6.73

Above output has negative and positive values which we can't expect as the output of a probability distribution. We can use the exponential function to convert these values to positive values, and then we can normalize them to get a probability distribution.

In our Bigram we had counts and then probabilities, but for NN, we can't have direct integers, since gradient adjustment requires float values, so we do this. 

In [34]:
(xenc @ W).exp()

tensor([[0.7722, 0.4846, 0.7400, 0.3021, 1.3868, 0.3096, 3.4153, 4.4555, 5.7514,
         0.2838, 2.3977, 0.2795, 3.6020, 6.9761, 1.3586, 1.0583, 0.3641, 2.3422,
         4.1802, 1.6378, 2.8680, 2.2579, 1.7880, 3.0119, 0.3291, 0.5707, 0.5898],
        [0.5913, 0.3226, 0.6800, 2.6652, 0.8558, 1.3315, 3.6006, 0.4632, 0.8449,
         0.9178, 0.5136, 4.1375, 1.2661, 0.5186, 6.3973, 2.0969, 0.0969, 1.0385,
         0.2966, 0.7863, 1.3508, 0.4111, 1.7085, 1.5504, 5.9824, 1.2095, 0.8147],
        [4.8544, 0.4172, 0.5973, 1.2291, 0.9125, 3.4846, 0.5858, 1.8385, 0.2748,
         1.1000, 0.8560, 1.5813, 0.4852, 0.4223, 0.5030, 1.0103, 1.4501, 2.7963,
         1.1483, 0.1892, 1.1361, 0.4544, 0.7130, 0.6766, 1.0178, 0.3446, 2.0417],
        [4.8544, 0.4172, 0.5973, 1.2291, 0.9125, 3.4846, 0.5858, 1.8385, 0.2748,
         1.1000, 0.8560, 1.5813, 0.4852, 0.4223, 0.5030, 1.0103, 1.4501, 2.7963,
         1.1483, 0.1892, 1.1361, 0.4544, 0.7130, 0.6766, 1.0178, 0.3446, 2.0417],
        [3.4780, 0.2355,

In [36]:
logits = xenc @ W # log-counts 
counts = logits.exp() # counts equivalent to N, sort of
probs = counts / counts.sum(1, keepdim= True) # prob for next char. Also, these 2 lines are softmax
probs

tensor([[0.0607, 0.0100, 0.0123, 0.0042, 0.0168, 0.0123, 0.0027, 0.0232, 0.0137,
         0.0313, 0.0079, 0.0278, 0.0091, 0.0082, 0.0500, 0.2378, 0.0603, 0.0025,
         0.0249, 0.0055, 0.0339, 0.0109, 0.0029, 0.0198, 0.0118, 0.1537, 0.1459],
        [0.0290, 0.0796, 0.0248, 0.0521, 0.1989, 0.0289, 0.0094, 0.0335, 0.0097,
         0.0301, 0.0702, 0.0228, 0.0115, 0.0181, 0.0108, 0.0315, 0.0291, 0.0045,
         0.0916, 0.0215, 0.0486, 0.0300, 0.0501, 0.0027, 0.0118, 0.0022, 0.0472],
        [0.0312, 0.0737, 0.0484, 0.0333, 0.0674, 0.0200, 0.0263, 0.0249, 0.1226,
         0.0164, 0.0075, 0.0789, 0.0131, 0.0267, 0.0147, 0.0112, 0.0585, 0.0121,
         0.0650, 0.0058, 0.0208, 0.0078, 0.0133, 0.0203, 0.1204, 0.0469, 0.0126],
        [0.0312, 0.0737, 0.0484, 0.0333, 0.0674, 0.0200, 0.0263, 0.0249, 0.1226,
         0.0164, 0.0075, 0.0789, 0.0131, 0.0267, 0.0147, 0.0112, 0.0585, 0.0121,
         0.0650, 0.0058, 0.0208, 0.0078, 0.0133, 0.0203, 0.1204, 0.0469, 0.0126],
        [0.0150, 0.0086,

In [35]:
probs[0].sum(), probs.shape, probs[0]

(tensor(1.0000),
 torch.Size([5, 27]),
 tensor([0.0607, 0.0100, 0.0123, 0.0042, 0.0168, 0.0123, 0.0027, 0.0232, 0.0137,
         0.0313, 0.0079, 0.0278, 0.0091, 0.0082, 0.0500, 0.2378, 0.0603, 0.0025,
         0.0249, 0.0055, 0.0339, 0.0109, 0.0029, 0.0198, 0.0118, 0.1537, 0.1459]))

Let's look at just first row, it says, when "." is given as input to NN, the output we get is probabilities of the next character being "a", "b", "c", ..., "z". We can see that the probability of the next character being "a" is 0.1, "b" is 0.2, and so on.

As we tune W, our probs change, and we want to get it to as accurate ys. 

Finished Day 7 on 5/31/2026 at 18:01 -> video time (1:26:18)

Starting Day 8 on 6/1/2026 at 18:12

In [ ]:
# example 
nlls = torch.zeros(5)
for i in range(5):
    # i-th bigram
    x = xs[i].item()
    y = ys[i].item()
    print('-------')
    print(f'bigram: {i+1}, {itos[x]}, {itos[y]} (indexes {x}, {y})')
    print('input to the NN: ', x)
    print('output of the NN: ', probs[i])
    p = probs[i, y]
    print(f'probability of the correct next char: {p.item()}')
    logp = torch.log(p)
    print(f'log probability of the correct next char: {logp.item()}')
    nll = -logp
    print(f'negative log likelihood of the correct next char: {nll.item()}')
    nlls[i] = nll 
    
print('========')
print(f'average negative log likelihood: {nlls.mean().item()}')

In [40]:
# The average is pretty high and we want to minimize it. 
# ------ Optimization --------
xs, ys

(tensor([ 0,  5, 13, 13,  1]), tensor([ 5, 13, 13,  1,  0]))

In [46]:
# randomly initialize weights of NN
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator= g, requires_grad= True)

In [57]:
# forward pass
xenc = F.one_hot(xs, num_classes = 27).float()
logits = xenc @ W
counts = logits.exp()
probs = counts / counts.sum(1, keepdim = True)
# we are interested in the probabilities of the correct next char, which is given by probs[range(len(xs)), ys]
loss = -probs[torch.arange(len(xs)), ys].log().mean()

In [58]:
loss

tensor(3.7292, grad_fn=<NegBackward0>)

In [55]:
# backward pass
W.grad = None # set grad to zero 
loss.backward()

In [56]:
# update weights
W.data += -0.1 * W.grad # learning rate is 0.1, you can change it to see how it affects the training process.

In [63]:
# for real this time
xs, ys = [], []
for w in words[:]:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)
xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print('number of examples: ', num) 

# initialize the network 
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator= g, requires_grad= True)

number of examples:  228146


In [ ]:
# gradient descent 
for k in range(100):
    # forward pass
    xenc  = F.one_hot(xs, num_classes = 27).float()
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim = True)
    loss = -probs[torch.arange(len(xs)), ys].log().mean() + 0.01 * (W**2).mean()
    # L2 regularization to prevent overfitting, you can change the coefficient to see how it affects the training process.
    # more emphasis on regularization -> more generalization, less overfitting. It smooths out the probs, just like (N+1) did.
    print("Loss: ", loss.item())
    # backward pass
    W.grad = None # set grad to zero
    loss.backward()
    
    # update weights
    W.data += -50 * W.grad 

Finished Day 8 on 6/1/2026 at 19:08 - Video Done. Onto more complex models!